In [ ]:
%pip install yfinance statsmodels pandas numpy scipy scikit-learn

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import mean_absolute_error

In [4]:
# 1. FETCH MARKET DATA
# Pulling the exact indices required
tickers = ["^GSPC", "^DJI", "^IXIC", "^RUT", "^TNX", "^VIX"]
# Download and select the 'Close' price specifically
raw_data = yf.download(tickers, start="1998-01-01", end="2020-04-01")['Close'].dropna()

[*********************100%***********************]  6 of 6 completed


In [5]:
# 2. DATA PREPARATION
# Log-transforming the prices
data = np.log(raw_data)
y = data["^GSPC"]  # S&P 500 Target
X = data[["^DJI", "^IXIC", "^RUT", "^TNX", "^VIX"]]  # Predictors

In [6]:
def run_stress_test(train_end, shock_start, shock_end):
    # Split into 'Normal' training and 'Shock' testing periods
    train_y, train_X = y[:train_end], X[:train_end]
    test_y, test_X = y[shock_start:shock_end], X[shock_start:shock_end]

    # 3. ARIMA(1,1,1) MODEL WITH EXOGENOUS REGRESSORS (ARIMAX)
    # d=1 handles non-stationarity automatically via differencing
    model = sm.tsa.ARIMA(train_y, order=(1, 1, 1), exog=train_X)
    results = model.fit()

    # 4. FORECASTING & BACK-TRANSFORMATION
    forecast = results.get_forecast(steps=len(test_y), exog=test_X)
    predicted = np.exp(forecast.summary_frame()['mean'])
    actual = np.exp(test_y)

    # 5. TEST WHETHER THE FORECAST FAILS
    # Paired t-test of forecast against actual values over shock window
    t_stat, p_val = stats.ttest_rel(predicted, actual)
    mae = mean_absolute_error(actual, predicted) # Quantifying the "Miss"
    return p_val, mae

In [7]:
# EXECUTION
p_2008, mae_2008 = run_stress_test('2008-08-31', '2008-09-29', '2008-10-10')
p_2020, mae_2020 = run_stress_test('2020-01-31', '2020-02-24', '2020-03-06')

c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_ind

In [8]:
print(f"2008 Shock: P-Value = {p_2008:.10f} | MAE = {mae_2008:.2f} Index Points")
print(f"2020 Shock: P-Value = {p_2020:.10f} | MAE = {mae_2020:.2f} Index Points")

2008 Shock: P-Value = 0.0000008327 | MAE = 30.09 Index Points
2020 Shock: P-Value = 0.0000000610 | MAE = 24.47 Index Points


In [9]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson


In [10]:
# 1. ADF test (stationarity): log prices vs daily changes
adf_level = adfuller(y)
adf_diff = adfuller(y.diff().dropna())
print("ADF log prices: p-value =", round(adf_level[1], 4))
print("ADF daily changes: p-value =", round(adf_diff[1], 4))

# 2. Breusch-Pagan test (heteroskedasticity): raw prices vs log prices
raw_fit = sm.OLS(raw_data["^GSPC"], sm.add_constant(raw_data[X.columns])).fit()
log_fit = sm.OLS(y, sm.add_constant(X)).fit()
print("Breusch-Pagan raw prices: p-value =", round(het_breuschpagan(raw_fit.resid, raw_fit.model.exog)[1], 4))
print("Breusch-Pagan log prices: p-value =", round(het_breuschpagan(log_fit.resid, log_fit.model.exog)[1], 4))

# 3. Durbin-Watson test (autocorrelation): residuals of the model fitted before 2008
model_2008 = sm.tsa.ARIMA(y[:"2008-08-31"], order=(1, 1, 1), exog=X[:"2008-08-31"]).fit()
print("Durbin-Watson =", round(durbin_watson(model_2008.resid.iloc[1:]), 2))

ADF log prices: p-value = 0.8249
ADF daily changes: p-value = 0.0
Breusch-Pagan raw prices: p-value = 0.0
Breusch-Pagan log prices: p-value = 0.0


c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Akshath\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Durbin-Watson = 2.0
